In [ ]:
# 1. Setup and Imports
import os
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

In [ ]:
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 2. Define Paths and Age Ranges
# Assume your HPC dataset is organized as:
# path
#    0-10/
#    11-20/
#    21-30/
#    ... (folders named by age range)
path = kagglehub.dataset_download("aiolapo/fgnet-dataset")
data_dir = path
age_ranges = sorted(os.listdir(data_dir))
num_classes = len(age_ranges)
print(f"Found age ranges (classes): {age_ranges}")

In [ ]:
# 3. Data Transforms
# Resize images, convert to tensor, normalize using ImageNet stats
transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# 4. Datasets and DataLoaders
train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform)
val_dataset   = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=4)

In [ ]:
# 5. Build the Model
# Load pretrained ResNet-18
model = models.resnet18(pretrained=True)

# Replace final layer to match our number of age ranges
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model = model.to(device)

In [ ]:
# 6. Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# 7. Training and Validation Loop
def train_one_epoch(epoch):
    model.train()
    running_loss, running_corrects = 0.0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc  = running_corrects.double() / len(train_dataset)
    print(f"Epoch {epoch} Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

In [ ]:
def validate(epoch):
    model.eval()
    val_loss, val_corrects = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
    epoch_loss = val_loss / len(val_dataset)
    epoch_acc  = val_corrects.double() / len(val_dataset)
    print(f"Epoch {epoch} Val   Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

In [ ]:
# 8. Run Training
num_epochs = 5
for epoch in range(1, num_epochs+1):
    train_one_epoch(epoch)
    validate(epoch)

In [ ]:
# 9. Save the Model
os.makedirs('checkpoints', exist_ok=True)
torch.save(model.state_dict(), 'checkpoints/resnet18_age_baseline.pth')
print("Training complete. Model saved to checkpoints/resnet18_age_baseline.pth")